# Pipeline Cost Estimation

Estimates token usage and cost for each stage of the evaluation dataset creation pipeline:
1. **Ground-truth generation** — input + output
2. **Transcript generation** — input + output
3. **Validation** — input only (no structured output cost tracked)

Tokens are counted with `tiktoken` using the `o200k_base` encoding (GPT-4o / GPT-5 family).
Update the pricing table in the **Pricing** cell if model costs change.

In [ ]:
import json
import sys
from pathlib import Path

import tiktoken

sys.path.insert(0, str(Path(".").resolve()))

from src.eval.config import MODEL_CONFIGS
from src.eval.utils import load_prompt_pair
from src.eval.prompts import schema_for_prompt, _suppressed_sections, _SECTION_TOPIC_LABELS
from src.eval.scenarios import SCENARIO_ARCHETYPES

In [2]:
MODEL_CONFIGS

{'ground_truth': {'model': 'gpt-5.4-mini',
  'temperature': 0.1,
  'reasoning': {'effort': 'none'},
  'timeout_seconds': 240.0},
 'transcript': {'model': 'gpt-5.4-mini',
  'temperature': 0.7,
  'reasoning': {'effort': 'none'},
  'timeout_seconds': 600.0},
 'validation': {'model': 'gpt-5.1',
  'temperature': 0.0,
  'reasoning': {'effort': 'high'},
  'timeout_seconds': 600.0}}

In [3]:
ENCODING = tiktoken.get_encoding("o200k_base")

def count_tokens(text: str) -> int:
    return len(ENCODING.encode(text))

print("tiktoken loaded, encoding:", ENCODING.name)

tiktoken loaded, encoding: o200k_base


## Load prompts and sample data

We use `medium_001` as a representative sample — it sits between easy and hard in terms of
JSON size and suppressed-section complexity.

In [ ]:
PROMPTS = {task: load_prompt_pair(task) for task in ("ground_truth", "transcript", "validation")}

SAMPLE_ID = "medium_001_self_employed_single"
gt_package = json.loads(Path(f"data/synthetic/ground_truth/{SAMPLE_ID}.json").read_text())
expected = gt_package["expected"]


transcript_text = Path(f"data/synthetic/transcripts/{SAMPLE_ID}.txt").read_text()

print(f"Sample: {SAMPLE_ID}")
print(f"  GT JSON chars  : {len(json.dumps(expected, indent=2)):,}")
print(f"  Transcript chars: {len(transcript_text):,}")

Sample: medium_001_self_employed_single
  GT JSON chars  : 8,224
  Transcript chars: 8,581


In [5]:
# Helper: build suppressed_sections string the same way make_transcript_prompt does
def make_suppressed_str(expected_json: dict) -> str:
    suppressed = _suppressed_sections(expected_json)
    if suppressed:
        lines = [f"  - {s}: {_SECTION_TOPIC_LABELS.get(s, s)}" for s in suppressed]
        return "\n" + "\n".join(lines)
    return "none"

## Stage 1 — Ground-truth generation

In [ ]:
archetype = next(a for a in SCENARIO_ARCHETYPES if a["id"] == gt_package["archetype_id"])
schema_json = json.dumps(schema_for_prompt(), indent=2)

gt_system = PROMPTS["ground_truth"]["system"]
gt_user = PROMPTS["ground_truth"]["user"].format(
    example_id=gt_package["example_id"],
    difficulty=gt_package["difficulty"],
    archetype_description=archetype["description"],
    section_targets=gt_package["section_targets"],
    challenge_tags=gt_package["challenge_tags"],
    age_band=gt_package["age_band"],
    include_mobile_phone=gt_package["include_mobile_phone"],
    include_email=gt_package["include_email"],
    client1_name=expected["client1"]["personal"]["first_name"] + " " + expected["client1"]["personal"]["last_name"],
    client2_name="generate a realistic unique name",
    schema_json=schema_json,
)

gt_output = json.dumps(expected, indent=2)

gt_sys_tokens   = count_tokens(gt_system)
gt_user_tokens  = count_tokens(gt_user)
gt_out_tokens   = count_tokens(gt_output)

print("Ground-truth generation")
print(f"  System prompt   : {gt_sys_tokens:>6,} tokens")
print(f"  User prompt     : {gt_user_tokens:>6,} tokens  (incl. full JSON schema)")
print(f"  Output (est.)   : {gt_out_tokens:>6,} tokens")
print(f"  Total input     : {gt_sys_tokens + gt_user_tokens:>6,} tokens")
print(f"  Model           : {MODEL_CONFIGS['ground_truth']['model']}")

Ground-truth generation
  System prompt   :    236 tokens
  User prompt     :  6,392 tokens  (incl. full JSON schema)
  Output (est.)   :  2,282 tokens
  Total input     :  6,628 tokens
  Model           : gpt-5.4-mini


## Stage 2 — Transcript generation

In [ ]:
tr_system = PROMPTS["transcript"]["system"]
tr_user = PROMPTS["transcript"]["user"].format(
    example_id=gt_package["example_id"],
    difficulty=gt_package["difficulty"],
    challenge_tags=gt_package["challenge_tags"],
    expected_json=json.dumps(expected, indent=2),
    suppressed_sections=make_suppressed_str(expected),
)

tr_sys_tokens   = count_tokens(tr_system)
tr_user_tokens  = count_tokens(tr_user)
tr_out_tokens   = count_tokens(transcript_text)

print("Transcript generation")
print(f"  System prompt   : {tr_sys_tokens:>6,} tokens")
print(f"  User prompt     : {tr_user_tokens:>6,} tokens  (incl. full GT JSON)")
print(f"  Output (est.)   : {tr_out_tokens:>6,} tokens")
print(f"  Total input     : {tr_sys_tokens + tr_user_tokens:>6,} tokens")
print(f"  Model           : {MODEL_CONFIGS['transcript']['model']}")

Transcript generation
  System prompt   :    316 tokens
  User prompt     :  3,413 tokens  (incl. full GT JSON)
  Output (est.)   :  2,536 tokens
  Total input     :  3,729 tokens
  Model           : gpt-5.4-mini


## Stage 3 — Validation

In [8]:
val_system = PROMPTS["validation"]["system"]
val_user = PROMPTS["validation"]["user"].format(
    example_id=gt_package["example_id"],
    difficulty=gt_package["difficulty"],
    challenge_tags=gt_package["challenge_tags"],
    expected_json=json.dumps(expected, indent=2),
    suppressed_sections=make_suppressed_str(expected),
    transcript=transcript_text,
)

val_sys_tokens   = count_tokens(val_system)
val_user_tokens  = count_tokens(val_user)

print("Validation")
print(f"  System prompt   : {val_sys_tokens:>6,} tokens")
print(f"  User prompt     : {val_user_tokens:>6,} tokens  (incl. GT JSON + transcript)")
print(f"  Total input     : {val_sys_tokens + val_user_tokens:>6,} tokens")
print(f"  Output          :  not tracked")
print(f"  Model           : {MODEL_CONFIGS['validation']['model']}")

Validation
  System prompt   :    933 tokens
  User prompt     :  5,505 tokens  (incl. GT JSON + transcript)
  Total input     :  6,438 tokens
  Output          :  not tracked
  Model           : gpt-5.1


## Cost estimation across all 80 examples

Update `PRICING` below with the actual per-million-token rates for your models.

In [ ]:
# this pricing from openai pricing API official page.
PRICING = {
    "gpt-5.4-mini": {"input": 0.75, "output": 4.50},
    "gpt-5.1":      {"input": 1.25, "output": 10.00},
}

N_EXAMPLES = 80

def cost_usd(tokens: int, model: str, direction: str) -> float:
    return tokens * PRICING[model][direction] / 1_000_000

stages = [
    {
        "name": "GT generation",
        "model": MODEL_CONFIGS["ground_truth"]["model"],
        "in_tokens":  gt_sys_tokens + gt_user_tokens,
        "out_tokens": gt_out_tokens,
    },
    {
        "name": "Transcript generation",
        "model": MODEL_CONFIGS["transcript"]["model"],
        "in_tokens":  tr_sys_tokens + tr_user_tokens,
        "out_tokens": tr_out_tokens,
    },
    {
        "name": "Validation",
        "model": MODEL_CONFIGS["validation"]["model"],
        "in_tokens":  val_sys_tokens + val_user_tokens,
        "out_tokens": 0,
    },
]

print(f"{'Stage':<25} {'Model':<16} {'Input tok':>10} {'Output tok':>11} {'In $':>9} {'Out $':>9} {'Total $':>9}")
print("-" * 95)

grand = 0.0
for s in stages:
    in_cost  = cost_usd(s["in_tokens"],  s["model"], "input")
    out_cost = cost_usd(s["out_tokens"], s["model"], "output")
    total    = in_cost + out_cost
    grand   += total
    print(
        f"{s['name']:<25} {s['model']:<16} "
        f"{s['in_tokens']:>10,} {s['out_tokens']:>11,} "
        f"${in_cost:>8.4f} ${out_cost:>8.4f} ${total:>8.4f}"
    )

print("-" * 95)
print(f"{'Per example':<52} ${grand:>8.4f}")
print(f"Total for {N_EXAMPLES} examples: ${grand * N_EXAMPLES:.4f}")


Stage                     Model             Input tok  Output tok      In $     Out $   Total $
-----------------------------------------------------------------------------------------------
GT generation             gpt-5.4-mini          6,628       2,282 $  0.0050 $  0.0103 $  0.0152
Transcript generation     gpt-5.4-mini          3,729       2,536 $  0.0028 $  0.0114 $  0.0142
Validation                gpt-5.1               6,438           0 $  0.0080 $  0.0000 $  0.0080
-----------------------------------------------------------------------------------------------
Per example                                          $  0.0375
Total for 80 examples: $2.9997
